## Problema de Dimensionamento de Lotes

Este notebook apresenta uma solução para o **Problema de Dimensionamento de Lotes com Capacidade (PDLC)**, uma aplicação clássica em planejamento de produção. O objetivo é determinar o quanto e quando produzir de múltiplos itens ao longo de um horizonte de tempo, visando minimizar os custos totais, que podem incluir custos de produção, estoque e preparação de maquinário.

A abordagem utilizada aqui emprega a modelagem de otimização matemática com o auxílio da biblioteca Gurobi em Python para encontrar a política de produção ótima.

### Passo 1: Instalação do Gurobi

Para resolver o modelo de otimização, primeiramente precisamos garantir que a biblioteca Gurobi esteja instalada no ambiente. O comando abaixo realiza a instalação de forma silenciosa (`-q`).

In [ ]:
# Instala a biblioteca Gurobi para Python
%pip install -q gurobipy

### Passo 2: Definição dos Dados do Problema

Nesta etapa, definimos todos os parâmetros que caracterizam o problema. Isso inclui o número de produtos, o horizonte de tempo, as demandas, os custos e as capacidades.

* $T$: Número de períodos no horizonte de planejamento.
* $n$: Quantidade de itens diferentes a serem produzidos.
* $I_0$: Lista com o estoque inicial para cada item.
* $sp$: Custos de preparação (setup) para cada item. Este é o custo incorrido para preparar a linha de produção para um determinado item.
* $h$: Custos de manutenção de estoque por item por período.
* $s$: Tempo de preparação (setup) consumido para cada item.
* $C$: Custo de produção unitário em cada período.
* $b$: Tempo de produção unitário para cada item.
* $CAP$: Capacidade de produção total disponível em cada período.
* $D$: Matriz de demanda, onde $D[i][t]$ é a demanda do item $i$ no período $t$.

In [ ]:
# Importação das bibliotecas necessárias
from gurobipy import Model, GRB
import random

# --- Geração de Dados para Instância Grande ---

# Use uma semente aleatória para garantir que os resultados sejam reprodutíveis
random.seed(42)

# Dimensões do problema
T = 40  # Períodos de tempo
n = 100  # Número de itens

# Estoque inicial (geralmente zero)
I0 = [0] * n

# Geração de custos e tempos aleatórios dentro de faixas realistas
sp = [random.randint(800, 1500) for _ in range(n)]  # Custos de preparação (setup)
h = [random.randint(10, 30) for _ in range(n)]      # Custos de manutenção de estoque
C = [random.randint(40, 70) for _ in range(T)]      # Custos de produção
s = [random.randint(50, 150) for _ in range(n)]     # Tempos de preparação (setup)
b = [random.randint(2, 10) for _ in range(n)]       # Tempos de produção por unidade

# Geração de uma matriz de demanda esparsa e irregular
D = [[0] * T for _ in range(n)]
for i in range(n):
    for t in range(T):
        # Define uma probabilidade de 40% de haver demanda em um dado período
        if random.random() < 0.4:
            D[i][t] = random.randint(20, 100)

# --- Cálculo da Capacidade ---
# Esta é a parte mais importante para garantir que o problema seja factível.
# 1. Calcule o tempo total de produção necessário para toda a demanda.
total_production_time = sum(D[i][t] * b[i] for i in range(n) for t in range(T))

# 2. Estime um tempo total de setup razoável (ex: cada item é preparado 6 vezes em média)
avg_setups_per_item = 6
estimated_total_setup_time = sum(s[i] * avg_setups_per_item for i in range(n))

# 3. Calcule a carga de trabalho total e a capacidade média necessária por período.
total_workload = total_production_time + estimated_total_setup_time
average_capacity_needed = total_workload / T

# 4. Defina a capacidade de cada período como um valor acima da média para dar
#    alguma folga, mas ainda manter o problema restrito.
capacity_per_period = int(average_capacity_needed * 1.40) # 40% de folga
CAP = [capacity_per_period] * T

print(f"Instância gerada com T={T}, n={n}")
print(D)
print(f"Capacidade definida por período: {capacity_per_period}")

### Passo 3: Inicialização do Modelo

O primeiro passo para a implementação é criar um objeto de modelo, que servirá como um contêiner para nossas variáveis, restrições e objetivo.

In [ ]:
# --- Construção do Modelo ---

# Criar o objeto do modelo
modelo = Model("Dimensionamento_de_Lotes")

### Passo 4: Definição das Variáveis de Decisão

As variáveis de decisão são as incógnitas do problema que o otimizador irá determinar.

* $X_{it}$: Uma variável contínua que representa a quantidade do item $i$ a ser produzida no período $t$.
* $I_{it}$: Uma variável contínua que representa o nível do estoque do item $i$ ao final do período $t$.
* $Y_{it}$: Uma variável binária que assume valor $1$ se ocorre a preparação para a produção do item $i$ no período $t$, e $0$ caso contrário.

In [ ]:
# Adicionar as variáveis de decisão ao modelo

# I[i,t]: Estoque do item i no final do período t
I = modelo.addVars(n, T, lb=0, name="I")

# X[i,t]: Quantidade produzida do item i no período t
X = modelo.addVars(n, T, lb=0, name="X")

# Y[i,t]: Variável binária (1 se houver produção de i em t, 0 caso contrário)
Y = modelo.addVars(n, T, vtype=GRB.BINARY, name="Y")

### Passo 5: Definição da Função Objetivo

A função objetivo define a meta do problema. Neste caso, queremos minimizar a soma de todos os custos envolvidos: manutenção de estoque, produção e preparação de maquinário.

**Fórmula Matemática:**
$$\text{Minimizar } Z = \sum_{i=0}^{n-1} \sum_{t=0}^{T-1} (h_i \cdot I_{it}) + \sum_{i=0}^{n-1} \sum_{t=0}^{T-1} (C_t \cdot X_{it}) + \sum_{i=0}^{n-1} \sum_{t=0}^{T-1} (sp_i \cdot Y_{it})$$

**Implementação em Gurobi:**

In [ ]:
# Definir a função objetivo do modelo
modelo.setObjective(
    sum(h[i] * I[i, t] for i in range(n) for t in range(T)) +
    sum(C[t] * X[i, t] for i in range(n) for t in range(T)) +
    sum(sp[i] * Y[i, t] for i in range(n) for t in range(T)),
    GRB.MINIMIZE
)

### Passo 6: Adição das Restrições

As restrições são as regras e limitações que a solução final deve respeitar. Vamos adicionar cada família de restrições separadamente.

#### 6.1 Restrição de Balanço de Estoque

Esta restrição garante a conservação do fluxo de material. O estoque no final de um período é o resultado do estoque anterior, mais a produção atual, menos a demanda atual.

**Fórmula Matemática:**
$$I_{it} = I_{i,t-1} + X_{it} - D_{it} \quad \forall i, t > 0$$
$$I_{i0} = I0_i + X_{i0} - D_{i0} \quad \forall i, t = 0$$

**Implementação em Gurobi:**

In [ ]:
# Adicionar a restrição de Balanço de Estoque
for i in range(n):
    for t in range(T):
        if t == 0:
            # Para o primeiro período, usa o estoque inicial I0
            modelo.addConstr(I[i, t] == I0[i] + X[i, t] - D[i][t], f"balanco_estoque_{i}_{t}")
        else:
            # Para os demais períodos, usa o estoque do período anterior I[i, t-1]
            modelo.addConstr(I[i, t] == I[i, t-1] + X[i, t] - D[i][t], f"balanco_estoque_{i}_{t}")

#### 6.2 Restrição de Capacidade de Produção

Garante que o tempo total consumido em cada período (soma dos tempos de produção e preparação) não exceda a capacidade disponível.

**Fórmula Matemática:**
$$\sum_{i=0}^{n-1} (b_i \cdot X_{it}) + \sum_{i=0}^{n-1} (s_i \cdot Y_{it}) \le CAP_t \quad \forall t$$

**Implementação em Gurobi:**

In [ ]:
# Adicionar a restrição de Limite de Capacidade
for t in range(T):
    modelo.addConstr(
        sum(b[i] * X[i, t] for i in range(n)) + sum(s[i] * Y[i, t] for i in range(n)) <= CAP[t],
        f"capacidade_{t}"
    )

#### 6.3 Restrição de Vínculo entre Produção e Preparação (Big-M)

Esta é uma restrição lógica fundamental: a produção de um item ($X_{it} > 0$) só pode acontecer se a linha de produção for preparada para ele ($Y_{it} = 1$). Usamos um número grande, "M", para modelar isso.

**Fórmula Matemática:**
$$X_{it} \le M \cdot Y_{it} \quad \forall i, t$$

**Implementação em Gurobi:**

In [ ]:
# Adicionar a restrição de Vínculo Produção-Preparação (Big-M)
# O valor 1000 atua como um "M" grande o suficiente para não limitar a produção.
for i in range(n):
    for t in range(T):
        modelo.addConstr(X[i, t] <= Y[i, t] * 1000, f"producao_{i}_{t}")

### Passo 7: Resolução do Modelo

Com o modelo completamente definido, o próximo passo é invocar o solver Gurobi para encontrar a solução ótima.

In [ ]:
# --- Resolução do Modelo ---

# Desliga o pré-processador do Gurobi.
# O presolve simplifica o modelo antes da otimização.
#modelo.setParam('Presolve', 0)
#print("Pré-processador (Presolve) do Gurobi DESLIGADO.")

# Desliga todas as heurísticas do Gurobi
#modelo.setParam('Heuristics', 0)
#print("Heurísticas do Gurobi DESLIGADAS.")

# Desliga todos os tipos de cortes (cutting planes)
#modelo.setParam('Cuts', 0)
#print("Cortes (Cutting Planes) do Gurobi DESLIGADOS.")
#print("-" * 30)

# Chama o solver do Gurobi para otimizar o modelo
modelo.optimize()

### Passo 8: Análise dos Resultados

Após a otimização, verificamos o status da solução. Se uma solução ótima foi encontrada, extraímos e exibimos os valores das variáveis de decisão, que compõem o plano de produção e estoque.

In [ ]:
# --- Exibição dos Resultados ---

# Verifica se o modelo encontrou uma solução ótima
if modelo.status == GRB.OPTIMAL:
    print("Solução ótima encontrada com custo total de: {:.2f}".format(modelo.objVal))
    print("-" * 60)
    for t in range(T):
        print(f"\n>>> PERÍODO {t} <<<")
        for i in range(n):
            # Acessa o valor da variável na solução com o atributo .X
            producao = X[i, t].X
            estoque = I[i, t].X
            preparacao = Y[i, t].X

            if preparacao > 0.5:  # Se houve preparação (Y=1)
                print(f"  Item {i}: Produzir: {producao:.1f} unidades Estoque Final: {estoque:.1f} unidades")
            else:
                print(f"  Item {i}: Nenhuma produção. Estoque Final: {estoque:.1f} unidades.")
else:
    print("Solução ótima não encontrada.")